# Transformer Confusion Matrices (in-domain)

This notebook runs inference on the test set for all 21 transformer models
(DistilBERT/BERT/RoBERTa × 7 configs) and produces a 3×3 confusion matrix for each.

**Drive layout (input):**
```
MyDrive/PATH/TO/
  data/splits/                                # train/val/test CSVs
  models/{distilbert,bert,roberta}/{config}/  # save_pretrained files
```

**Output:** `MyDrive/PATH/TO/results/confusion_matrices/{model}__{source_config}.npy`

Everything is zipped at the end → download and extract under `results/confusion_matrices/` in the project.

GPU: T4 is enough. Total time ~30-45 min.

In [ ]:
!pip install -q transformers

from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = "/content/drive/MyDrive/PATH/TO"  # TODO: set to your Google Drive project path
SPLITS_DIR = f"{DRIVE_BASE}/data/splits"
MODELS_DIR = f"{DRIVE_BASE}/models"
CM_DIR = f"{DRIVE_BASE}/results/confusion_matrices"
os.makedirs(CM_DIR, exist_ok=True)

for p in (SPLITS_DIR, MODELS_DIR):
    assert os.path.isdir(p), f"MISSING: {p}"
print(f"Outputs will go to: {CM_DIR}")

In [ ]:
import csv
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}

TRANSFORMERS = ["distilbert", "bert", "roberta"]
CONFIGS = [
    "twitter", "skytrax", "airlinequality",
    "twitter+skytrax", "twitter+airlinequality", "skytrax+airlinequality",
    "twitter+skytrax+airlinequality",
]
MAX_LEN_TWEET = 128
MAX_LEN_REVIEW = 512
BATCH = 32  # inference can use bigger batch than training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def load_test(source_config):
    path = os.path.join(SPLITS_DIR, f"{source_config}_test.csv")
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(LABEL_MAP[row["label"]])
    return texts, np.array(labels)


class InferenceDS(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts, self.tokenizer, self.max_len = texts, tokenizer, max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length",
                             max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}

print("Helpers ready.")

In [ ]:
def compute_cm(model_key, source_config):
    out = os.path.join(CM_DIR, f"{model_key}__{source_config}.npy")
    if os.path.exists(out):
        return "SKIP"

    model_dir = os.path.join(MODELS_DIR, model_key, source_config)
    if not os.path.isdir(model_dir):
        return f"MISS ({model_dir})"

    max_len = MAX_LEN_TWEET if source_config == "twitter" else MAX_LEN_REVIEW
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    texts, y_true = load_test(source_config)
    loader = DataLoader(InferenceDS(texts, tokenizer, max_len), batch_size=BATCH)

    preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attn).logits
            preds.append(logits.argmax(1).cpu().numpy())
    y_pred = np.concatenate(preds)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    np.save(out, cm)

    # Free GPU
    del model
    torch.cuda.empty_cache()
    return "OK"


total = len(TRANSFORMERS) * len(CONFIGS)
done = 0
for m in TRANSFORMERS:
    for cfg in CONFIGS:
        done += 1
        try:
            status = compute_cm(m, cfg)
        except Exception as e:
            status = f"FAIL ({type(e).__name__}: {e})"
        print(f"  [{done:2d}/{total}] {status:6s}  {m} | {cfg}")

print("\nDone.")

## Zip + Download

21 small .npy files — download as a single zip and extract under `results/confusion_matrices/` in the project.

In [ ]:
import subprocess
zip_path = f"{DRIVE_BASE}/transformer_cms.zip"
subprocess.run(["zip", "-j", zip_path] + [
    os.path.join(CM_DIR, fn) for fn in sorted(os.listdir(CM_DIR))
    if fn.endswith(".npy") and any(fn.startswith(m + "__") for m in TRANSFORMERS)
], check=True)
print(f"\nZip: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"(google.colab.files not available: {e}) -- download from the Drive UI.")